# Corrosion Preprocessing — Durability Target

Builds `data/composition_durability.csv`, pairing the Pt/Pd/Au/Ir alloy composition (`PtPdAuIr_summary.csv`) with a durability target derived from `corrosion.xlsx`.

**Target.** The corrosion sheet reports element mass before and after a 3-hour chronoamperometry hold. The durability target is the mass change, final (3Hr) − initial, in mg/cm² per element (`dm_Pt … dm_Si`), plus `dm_total_abs`, the total absolute alloy mass change as a single durability magnitude. Negative `dm` means net dissolution.

> **Scope caveat.** The dataset is small and is just for analysis more than prediction.

In [1]:
import pandas as pd

from oer_utils import (
    load_catalyst_summary,
    match_composition,
    composition_values,
)

# ── Paths ────────────────────────────────────────────────────
CORROSION_XLSX = 'data/external/corrosion.xlsx'
SUMMARY_CSV = 'data/PtPdAuIr_summary.csv'
OUTPUT_CSV = 'data/composition_durability.csv'

# Elements measured in the corrosion sheet. Pt/Pd/Au/Ir are the alloy;
# Si is the substrate, carried through but flagged (see below).
ELEMENTS = ['Pt', 'Pd', 'Au', 'Ir', 'Si']

## 1. Read the corrosion sheet

In [2]:
def load_mass_change(path):
    """Read the stacked corrosion sheet into one row per run."""
    raw = pd.read_excel(path, sheet_name=0, header=None)
    raw = raw.rename(columns={0: 'run', 1: 'rowtype'})
    raw['run'] = raw['run'].ffill()

    is_diff = (
        raw['rowtype'].astype(str).str.strip().str.lower() == 'difference'
    )
    diff = raw[is_diff].set_index('run')[[2, 3, 4, 5, 6]]
    diff.columns = ELEMENTS
    diff.index.name = 'run'
    return diff.dropna(how='all')


mass_change = load_mass_change(CORROSION_XLSX)
print(f'Runs with corrosion data: {len(mass_change)}')
print('\nMass change (final 3Hr - initial), mg/cm²:')
print(mass_change.round(4))

Runs with corrosion data: 6

Mass change (final 3Hr - initial), mg/cm²:
              Pt      Pd      Au      Ir      Si
run                                             
run9-22  -0.0013  0.0006 -0.0033 -0.0004   0.013
run9-34  -0.0012  0.0007 -0.0003 -0.0006    0.01
run10-9   -0.001 -0.0024 -0.0007 -0.0024  0.0338
run10-34       0  0.0028 -0.0027 -0.0003  0.0074
run11-12  0.0006  0.0012 -0.0051 -0.0007  0.0068
run11-18 -0.0004  0.0018 -0.0065 -0.0042  0.0261


## 2. Join composition features

In [3]:
# ── Join composition features to the corrosion target ────────
catalyst_df = load_catalyst_summary(SUMMARY_CSV)

records = []
for run, row in mass_change.iterrows():
    cat_row = match_composition(catalyst_df, run)
    if len(cat_row) != 1:
        print(f'  {run}: matched {len(cat_row)} rows — skipping.')
        continue

    Pt, Pd, Au, Ir = composition_values(cat_row)

    # Differences of the sheet's 4-decimal masses; round to that precision
    # so the CSV does not imply resolution the instrument did not have.
    dm = {f'dm_{el}': round(float(row[el]), 4) for el in ELEMENTS}
    # Overall durability magnitude: total alloy mass lost/gained. Si is the
    # substrate, so it is excluded from this catalyst-durability summary.
    alloy = ['Pt', 'Pd', 'Au', 'Ir']
    dm_total_abs = round(float(sum(abs(row[el]) for el in alloy)), 4)

    records.append(
        {
            'run': run,
            'Pt': Pt,
            'Pd': Pd,
            'Au': Au,
            'Ir': Ir,
            **dm,
            'dm_total_abs': dm_total_abs,
        }
    )

durability = pd.DataFrame.from_records(records)
print(f'Joined rows: {len(durability)}')
durability

Joined rows: 6


,run,Pt,Pd,Au,Ir,dm_Pt,dm_Pd,dm_Au,dm_Ir,dm_Si,dm_total_abs
0,run9-22,0.3275,0.1510,0.4458,0.0758,-0.0013,0.0006,-0.0033,-0.0004,0.0130,0.0056
1,run9-34,0.6525,0.1073,0.2393,0.0009,-0.0012,0.0007,-0.0003,-0.0006,0.0100,0.0028
2,run10-9,0.1599,0.2450,0.4481,0.1471,-0.0010,-0.0024,-0.0007,-0.0024,0.0338,0.0065
3,run10-34,0.5665,0.0950,0.3169,0.0216,0.0000,0.0028,-0.0027,-0.0003,0.0074,0.0058
4,run11-12,0.0417,0.1253,0.3351,0.4979,0.0006,0.0012,-0.0051,-0.0007,0.0068,0.0076
5,run11-18,0.0628,0.1019,0.5044,0.3309,-0.0004,0.0018,-0.0065,-0.0042,0.0261,0.0129


## 3. Write the durability CSV

In [4]:
# ── Write composition_durability.csv ─────────────────────────
# Feature columns first (Pt, Pd, Au, Ir), then the per-element mass-change
# targets (dm_*) and the summary durability magnitude, mirroring the
# composition_*.csv layout used by the other preprocessing notebooks.
# The run label is kept only for traceability and is not a model feature.
column_order = (
    ['Pt', 'Pd', 'Au', 'Ir']
    + [f'dm_{el}' for el in ELEMENTS]
    + ['dm_total_abs', 'run']
)
durability = durability[column_order]
durability.to_csv(OUTPUT_CSV, index=False)

print(f'Wrote {OUTPUT_CSV}  ({len(durability)} rows)')
print(pd.read_csv(OUTPUT_CSV).round(4).to_string(index=False))

Wrote data/composition_durability.csv  (6 rows)
    Pt     Pd     Au     Ir   dm_Pt   dm_Pd   dm_Au   dm_Ir  dm_Si  dm_total_abs      run
0.3275 0.1510 0.4458 0.0758 -0.0013  0.0006 -0.0033 -0.0004 0.0130        0.0056  run9-22
0.6525 0.1073 0.2393 0.0009 -0.0012  0.0007 -0.0003 -0.0006 0.0100        0.0028  run9-34
0.1599 0.2450 0.4481 0.1471 -0.0010 -0.0024 -0.0007 -0.0024 0.0338        0.0065  run10-9
0.5665 0.0950 0.3169 0.0216  0.0000  0.0028 -0.0027 -0.0003 0.0074        0.0058 run10-34
0.0417 0.1253 0.3351 0.4979  0.0006  0.0012 -0.0051 -0.0007 0.0068        0.0076 run11-12
0.0628 0.1019 0.5044 0.3309 -0.0004  0.0018 -0.0065 -0.0042 0.0261        0.0129 run11-18
